# Bronchoscopy Multi-Agent System 

The system is separated into two lines:
1. Research pipeline:
    used for single-turn multi-agent reasoning and multi-turn JSON generation.
    It is better for structured outputs, explainability, and research demos.
2. Runtime pipeline:
    used for fast, stable, and concise real-time guidance.
    It is better suited for online interaction, UI display, and TTS.

### 为什么分成两条线路？
最初系统主要依赖 MAS 生成指导内容。MAS 更适合做丰富的多代理推理和结构化 JSON 输出，但在实时支气管镜引导里，它往往太慢，而且稳定性不够。
所以现在系统分成两条线：
1. Research pipeline：
    用于单轮多代理推理和多轮 JSON 生成，
    更适合结构化输出、可解释性和研究展示。
2. Runtime pipeline：
    用于快速、稳定、简短的实时指导生成，
    更适合在线交互、UI 展示和 TTS。

## How to run / 运行说明

This notebook uses the **source code directly** (no installation required).

1. Unzip the project folder.
2. Open this notebook inside the project root directory.
3. Run the next cell to add `src/` to the Python path.
4. Execute the remaining cells in order.
5. Python version: 3.11
6. Tested on macOS.

本 notebook 直接调用源码（无需安装）。

1. 解压项目文件夹。
2. 在项目根目录打开本 notebook。
3. 运行下一格代码，将 `src/` 加入 Python 路径。
4. 按顺序运行后续单元格即可。

---

If any import error occurs, please ensure the notebook is opened at the project root (the folder containing `src/`).

如出现导入错误，请确认 notebook 位于包含 `src/` 文件夹的项目根目录。


In [2]:
# --- Setup project path ---

import sys
from pathlib import Path

project_root = Path.cwd()
sys.path.insert(0, str(project_root / "src"))

print(sys.path[0])

/Users/gwendolynlestrange/Desktop/broncho_mas_final_clean/src


In [3]:
print("Testing core imports...")

from broncho_mas import SmolAgentsLLM

# Runtime path
from broncho_mas.runtime.runtime_manager import RuntimeManager

print("Core imports OK.")

Testing core imports...


/opt/anaconda3/envs/hf310/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Core imports OK.


In [9]:
import broncho_mas

# Print where broncho_mas is imported from / 打印 broncho_mas 的实际导入路径
print("broncho_mas imported from:", broncho_mas.__file__)

# Show available top-level names / 显示包顶层可用名称
print("Available top-level names:", [x for x in dir(broncho_mas) if not x.startswith("_")][:30])

# Assume research pipeline is available / 默认 research 流程可用
# This is to test the Multi-agent pipeline.
research_available = True

try:
    # Try to import the research manager / 尝试导入 research manager
    from broncho_mas.research.manager import MultiAgentManager
    print("Research imports OK.")
except Exception as e:
    # Mark as unavailable if import fails / 如果导入失败，则标记为不可用
    research_available = False
    print("Research pipeline not available in this environment.")
    print("Reason:", repr(e))

# Create runtime manager / 创建 runtime manager
# This is to test the real-time pipeline.
runtime = RuntimeManager()

# Print status / 打印状态
print("RuntimeManager created successfully.")
print(runtime)

broncho_mas imported from: /Users/gwendolynlestrange/Desktop/broncho_mas_final_clean/src/broncho_mas/__init__.py
Available top-level names: ['MultiAgentManager', 'RuntimeManager', 'SmolAgentsLLM', 'adapter', 'annotations', 'research', 'runtime', 'shared']
Research imports OK.
[runtime] provider=hf model=Qwen/Qwen2.5-14B-Instruct
RuntimeManager created successfully.


In [5]:
import os
import getpass
os.environ["BRONCHO_MODEL"] = "Qwen/Qwen2.5-14B-Instruct"
os.environ["BRONCHO_TOOL_CHOICE"] = "auto"
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN: ")

Enter HF_TOKEN:  ········


## Runtime-first synthetic scenarios

These examples use synthetic structured inputs to demonstrate:
- normal next-step coaching
- drift / reorientation style guidance
- progression through curriculum
- safe, concise runtime outputs

The exact output depends on your current implementation, but the notebook is designed to show whether the pipeline is wired and behaving sensibly.

In [ ]:
from pprint import pprint

def build_runtime_prompt(payload):
    current_situation = []

    current_situation.append(f"Current region: {payload.get('current_region')}")
    current_situation.append(f"Target region: {payload.get('target_region')}")
    current_situation.append(f"Regions seen: {payload.get('regions_seen')}")
    current_situation.append(f"Phase: {payload.get('phase')}")
    current_situation.append(f"Target visible: {payload.get('is_target_visible')}")
    current_situation.append(f"Centered: {payload.get('is_centered')}")
    current_situation.append(f"Stable: {payload.get('is_stable')}")
    current_situation.append(f"Drift detected: {payload.get('drift_detected')}")
    current_situation.append(f"Backtracking: {payload.get('backtracking')}")
    current_situation.append(f"Need LLM: {payload.get('need_llm')}")

    current_situation_text = "\n".join(current_situation)
    previous_msgs = payload.get("previous_msgs", "")
    student_question = payload.get("student_question", "")
    soft_prompt = payload.get("soft_prompt", "")

    prompt = f"""CURRENT_SITUATION:
{current_situation_text}

PREVIOUS_MSGS:
{previous_msgs}

STUDENT_QUESTION:
{student_question}

SOFT_PROMPT:
{soft_prompt}
"""
    return prompt

def run_runtime_case(case_name, payload, debug=False):
    print("=" * 80)
    print("CASE:", case_name)

    if debug:
        print("INPUT PAYLOAD:")
        pprint(payload)

    prompt = build_runtime_prompt(payload)

    if debug:
        print("\nFORMATTED PROMPT:")
        print(prompt)

    result = runtime.run(prompt)

    print("\nDISPLAY OUTPUT:")
    print(result.get("ui_text", "<no ui_text>"))

In [ ]:
case_1 = {
    "regions_seen": [],
    "current_region": None,
    "target_region": "RB1",
    "is_target_visible": False,
    "is_centered": False,
    "is_stable": False,
    "drift_detected": False,
    "backtracking": False,
    "phase": "navigation",
    "need_llm": True,
    "soft_prompt": "No target region has been reached yet. Guide the learner toward the next airway.",
    "previous_msgs": "",
    "student_question": ""
}

result_1 = run_runtime_case("Initial orientation", case_1)

In [ ]:
case_2 = {
    "regions_seen": ["RB1"],
    "current_region": "RB1",
    "target_region": "RB2",
    "is_target_visible": False,
    "is_centered": False,
    "is_stable": False,
    "drift_detected": True,
    "backtracking": False,
    "phase": "navigation",
    "need_llm": True,
    "soft_prompt": "The learner has drifted off target and needs concise reorientation guidance.",
}

result_2 = run_runtime_case("Drift / reorientation", case_2)

In [ ]:
case_3 = {
    "regions_seen": ["RB1", "RB2"],
    "current_region": "RB2",
    "target_region": "RB3",
    "is_target_visible": True,
    "is_centered": False,
    "is_stable": False,
    "drift_detected": False,
    "backtracking": False,
    "phase": "navigation",
    "need_llm": True,
    "soft_prompt": "The target is visible but not yet centered or stable. Give short coaching.",
}

result_3 = run_runtime_case("Visible but unstable target", case_3)

In [ ]:
case_4 = {
    "regions_seen": ["RB1", "RB2", "RB3"],
    "current_region": "RB3",
    "target_region": "RB3",
    "is_target_visible": True,
    "is_centered": True,
    "is_stable": True,
    "drift_detected": False,
    "backtracking": False,
    "phase": "inspection",
    "need_llm": True,
    "soft_prompt": "The learner has stabilized at the target. Confirm and transition to the next step.",
}

result_4 = run_runtime_case("Stable at target / transition", case_4)

## Adapter demo

This section checks the adapter-facing API used to bridge runtime logic and higher-level LLM-style calling.
It is useful for verifying that the package surface is stable even if the full research pipeline is not active.

In [ ]:
adapter = SmolAgentsLLM()
print("Adapter created successfully.")
print(adapter)

In [ ]:
public_methods = [x for x in dir(adapter) if not x.startswith("_")]
print("Public adapter attributes/methods:")
print(public_methods)

In [ ]:
test_prompt = "Guide the learner toward the next bronchial branch in one or two short sentences."

if hasattr(adapter, "ask"):
    try:
        print("adapter.ask(...) output:")
        print(adapter.ask(test_prompt))
    except Exception as e:
        print("adapter.ask failed:", repr(e))
else:
    print("adapter.ask not found.")

if hasattr(adapter, "ask_structured"):
    try:
        print("\nadapter.ask_structured(...) output:")
        pprint(adapter.ask_structured(test_prompt))
    except Exception as e:
        print("adapter.ask_structured failed:", repr(e))
else:
    print("adapter.ask_structured not found.")

## Research pipeline
This pipeline uses the research manager to organize multi-agent reasoning and generate structured guidance.
It is mainly used for richer, more explainable outputs in the research line.
Part 1: Single-turn reasoning
In a single turn, the research manager takes the current input and produces structured guidance for that moment.
Part 2: JSON generation across multiple turns
Across multiple turns, the pipeline generates JSON outputs that can be stored, tracked, and used to support continuous multi-turn interaction.

### Research 流程：
这个流程通过 research manager 组织多代理推理，并生成结构化指导内容。
它主要用于 research 线中更丰富、更可解释的输出。
第一部分：单轮推理
在单个 turn 中，research manager 根据当前输入生成该时刻的结构化指导。

第二部分：多轮 JSON 生成
在多个 turn 中，这个流程会持续生成 JSON 输出，用于记录、跟踪，并支持连续的多轮交互。

In [10]:
if research_available:
    try:
        manager = MultiAgentManager("Qwen/Qwen3.5-27B")
        print("MultiAgentManager created successfully.")
        print("manager type:", type(manager))
    except Exception as e:
        print("Failed to instantiate MultiAgentManager:")
        print(type(e).__name__, str(e))
else:
    print("Research pipeline unavailable in this environment.")

[broncho_mas manager] version=0.0.3-research research_mode=agentic provider=hf model=Qwen/Qwen3.5-27B
MultiAgentManager created successfully.
manager type: <class 'broncho_mas.research.manager.MultiAgentManager'>


In [ ]:
if research_available:
    prompt = """
The learner is navigating bronchoscopy training.
Current region: RB2
Target region: RB3
The target is not yet centered and minor drift is present.
Provide concise instructional guidance.
""".strip()

    try:
        if hasattr(manager, "run"):
            out = manager.run(prompt)
            print("UI text:")
            print(out.get("ui_text", "<no ui_text found>"))
        else:
            print("MultiAgentManager has no .run() method.")
    except Exception as e:
        print("Research prompt execution failed:")
        print(repr(e))
else:
    print("Skipping research prompt run.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ORCHESTRATOR RULES:                                                                                             │
│ - Call curriculum_progress_tool first.                                                                          │
│ - Then call landmark_lookup_tool.                                                                               │
│ - Finally output ONE JSON object via final_answer.                                                              │
│ - Output keys: curriculum_progress, landmark_hint.                                                              │
│                                                                                                                 │
│                                                                                                                 │
│ INPUTS:                                                                                                         │
│ - regions_seen_json: []                                                                                         │
│ - curriculum_order_json: ["RB1", "RB2", "RB3", "RB4", "RB5", "RB6", "RB7", "RB8", "RB9", "RB10", "LB1+2",       │
│ "LB3", "LB4", "LB5", "LB6", "LB8", "LB9", "LB10"\]                                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3.5-27B ───────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Step 1: Call curriculum_progress_tool                                                                          
  regions_seen = '[]'                                                                                              
  curriculum_order = '["RB1", "RB2", "RB3", "RB4", "RB5", "RB6", "RB7", "RB8", "RB9", "RB10", "LB1+2", "LB3",      
  "LB4", "LB5", "LB6", "LB8", "LB9", "LB10"]'                                                                      
                                                                                                                   
  progress_result = curriculum_progress_tool(regions_seen_json=regions_seen,                                       
  curriculum_order_json=curriculum_order)                                                                          
  print("Curriculum progress result:", progress_result)                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Curriculum progress result: {"reached": [], "missing": ["RB1", "RB2", "RB3", "RB4", "RB5", "RB6", "RB7", "RB8", 
"RB9", "RB10", "LB1+2", "LB3", "LB4", "LB5", "LB6", "LB8", "LB9", "LB10"], "reached_count": 0, "total": 18, 
"coverage_ratio": 0.0, "next_airway": "RB1"}

Out: None

[Step 1: Duration 7.58 seconds| Input tokens: 2,406 | Output tokens: 437]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Step 2: Call landmark_lookup_tool with the next airway from the curriculum progress                            
  next_airway = "RB1"                                                                                              
  landmark_result = landmark_lookup_tool(next_airway=next_airway)                                                  
  print("Landmark lookup result:", landmark_result)                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Landmark lookup result: {"anchor_landmark": "L2_RUL", "recommended_angle": "90° right", "recognition_cue": 
"Mercedes sign / trifurcation"}

Out: None

[Step 2: Duration 3.72 seconds| Input tokens: 5,357 | Output tokens: 618]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [11]:
# --- generate timeline.json for report for multi-turn output test ---

import json
from pathlib import Path

recording_dir = Path("./_simple_mock")
recording_dir.mkdir(exist_ok=True)

frames = [
    {"t_rel": 0.0, "anatomical_position": "RB1", "current_target": "RB1", "just_reached": True,  "reached_regions": ["RB1"]},
    {"t_rel": 2.0, "anatomical_position": "RB2", "current_target": "RB2", "just_reached": True,  "reached_regions": ["RB1","RB2"]},
    {"t_rel": 4.0, "anatomical_position": "RB2", "current_target": "back", "just_reached": False, "reached_regions": ["RB1","RB2"]},
    {"t_rel": 6.0, "anatomical_position": "RB3", "current_target": "RB3", "just_reached": True,  "reached_regions": ["RB1","RB2","RB3"]},
    {"t_rel": 8.0, "anatomical_position": "RB4", "current_target": "RB4", "just_reached": True,  "reached_regions": ["RB1","RB2","RB3","RB4"]},
]
(recording_dir / "timeline.json").write_text(json.dumps(frames, indent=2), encoding="utf-8")
print("timeline ready:", recording_dir / "timeline.json")

timeline ready: _simple_mock/timeline.json


In [ ]:
#--- multi-turn output test ---

def make_prompt(pos, target=None, reached=None, q=""):
    target = target or pos
    reached = reached or []
    return f"""
CURRENT_SITUATION:
anatomical_position={pos}
current_target={target}
just_reached=true
reached_regions(last)={reached}

PREVIOUS_MSGS:

STUDENT_QUESTION:
{q}
""".strip()

sequence = [
    ("RB1", "RB1", ["RB1"], ""),
    ("RB2", "RB2", ["RB1","RB2"], "Where should I go next?"),
    ("RB2", "back", ["RB1","RB2"], "I feel lost. Should I withdraw a bit?"),
    ("RB3", "RB3", ["RB1","RB2","RB3"], ""),
    ("RB4", "RB4", ["RB1","RB2","RB3","RB4"], "Can you show a diagram?"),
]

results = []
for i, (pos, tgt, reached, q) in enumerate(sequence, 1):
    out = manager.run(make_prompt(pos, tgt, reached, q))
    results.append(out)

    cp = out.get("curriculum_progress", {}) or {}
    instr = (out.get("instructor") or "").strip()
    vflag = out.get("needs_visual_guidance", False)

    # keep instructor preview short for notebook readability
    instr_preview = "\n".join(instr.splitlines()[:4])  # first 4 lines
    if len(instr.splitlines()) > 4:
        instr_preview += "\n..."

    print(f"\n--- Turn {i} ---")
    print("pos:", pos, "| target:", tgt)
    print("next_airway:", cp.get("next_airway"))
    print("coverage_ratio:", cp.get("coverage_ratio"))
    print("needs_visual_guidance:", vflag)
    print("instructor preview:\n", instr_preview)

print("\n===== FINAL REPORT =====")
print(manager.get_report(recording_dir=str(recording_dir)))

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ORCHESTRATOR RULES:                                                                                             │
│ - Call curriculum_progress_tool first.                                                                          │
│ - Then call landmark_lookup_tool.                                                                               │
│ - Finally output ONE JSON object via final_answer.                                                              │
│ - Output keys: curriculum_progress, landmark_hint.                                                              │
│                                                                                                                 │
│                                                                                                                 │
│ INPUTS:                                                                                                         │
│ - regions_seen_json: ["RB1"\]                                                                                   │
│ - curriculum_order_json: ["RB1", "RB2", "RB3", "RB4", "RB5", "RB6", "RB7", "RB8", "RB9", "RB10", "LB1+2",       │
│ "LB3", "LB4", "LB5", "LB6", "LB8", "LB9", "LB10"\]                                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3.5-27B ───────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Convert the input lists to JSON strings                                                                        
  regions_seen_json = '["RB1"]'                                                                                    
  curriculum_order_json = '["RB1", "RB2", "RB3", "RB4", "RB5", "RB6", "RB7", "RB8", "RB9", "RB10", "LB1+2",        
  "LB3", "LB4", "LB5", "LB6", "LB8", "LB9", "LB10"]'                                                               
                                                                                                                   
  # Step 1: Call curriculum_progress_tool                                                                          
  progress = curriculum_progress_tool(regions_seen_json=regions_seen_json,                                         
  curriculum_order_json=curriculum_order_json)                                                                     
  print(f"Curriculum Progress: {progress}")                                                                        
                                                                                                                   
  # Step 2: Determine next airway (the one after RB1 in the order, which is RB2)                                   
  next_airway = "RB2"                                                                                              
  landmark = landmark_lookup_tool(next_airway=next_airway)                                                         
  print(f"Landmark for {next_airway}: {landmark}")                                                                 
                                                                                                                   
  # Step 3: Create final answer as JSON                                                                            
  final_answer({"curriculum_progress": progress, "landmark_hint": landmark})                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Curriculum Progress: {"reached": ["RB1"], "missing": ["RB2", "RB3", "RB4", "RB5", "RB6", "RB7", "RB8", "RB9", 
"RB10", "LB1+2", "LB3", "LB4", "LB5", "LB6", "LB8", "LB9", "LB10"], "reached_count": 1, "total": 18, 
"coverage_ratio": 0.056, "next_airway": "RB2"}
Landmark for RB2: {"anchor_landmark": "L2_RUL", "recommended_angle": "90° right", "recognition_cue": "Mercedes sign
/ trifurcation"}

Final answer: {'curriculum_progress': '{"reached": ["RB1"], "missing": ["RB2", "RB3", "RB4", "RB5", "RB6", "RB7", 
"RB8", "RB9", "RB10", "LB1+2", "LB3", "LB4", "LB5", "LB6", "LB8", "LB9", "LB10"], "reached_count": 1, "total": 18, 
"coverage_ratio": 0.056, "next_airway": "RB2"}', 'landmark_hint': '{"anchor_landmark": "L2_RUL", 
"recommended_angle": "90° right", "recognition_cue": "Mercedes sign / trifurcation"}'}

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'submit_statistics' with arguments: {'trend': 'stable', 'likely_issue': 'Difficulty transitioning │
│ to next airway region', 'coach_focus_next': 'Maintain centerline alignment approaching RB4', 'notes': 'Student  │
│ successfully progressed through RB1→RB2→RB3 sequentially. Currently at RB3, ready for next milestone RB4. Low   │
│ coverage (16.7%) expected at early stages. Consistent pattern suggests steady learning without acceleration or  │
│ deceleration.'}                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"trend": "stable", "likely_issue": "Difficulty transitioning to next airway region", 
"coach_focus_next": "Maintain centerline alignment approaching RB4", "notes": "Student successfully progressed 
through RB1→RB2→RB3 sequentially. Currently at RB3, ready for next milestone RB4. Low coverage (16.7%) expected at 
early stages. Consistent pattern suggests steady learning without acceleration or deceleration."}

[Step 1: Duration 11.61 seconds| Input tokens: 1,778 | Output tokens: 627]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━